In [5]:
import sys
sys.path.append("..")

In [8]:
import tensorflow_hub as hub

In [9]:
import sys
sys.path.append("..")

from config import USE_MODEL_URL

print("Loading encoder...")

encoder = hub.load(USE_MODEL_URL)

print("Encoder loaded")

Loading encoder...



Encoder loaded


In [10]:
import pandas as pd
import numpy as np
from pathlib import Path
import tensorflow as tf
import tensorflow_hub as hub
from sklearn.model_selection import train_test_split

In [11]:
data_path = Path("../data/labeled/weak_labels.csv")

df = pd.read_csv(data_path)

print("Dataset size:", len(df))
df.head()

Dataset size: 600


,profile_text,template_index
0,Rahul Sharma is a Social Media Manager with 2 ...,11
1,Rohit Kumar is a Investment Analyst with 4 yea...,26
2,Anjali Gupta is a HR Manager with 10 years of ...,44
3,Rahul Sharma is a Backend Engineer with 2 year...,30
4,Vikas Mehta is a Talent Acquisition Specialist...,39


In [12]:
texts = df["profile_text"].values
labels = df["template_index"].values

print("Example text:", texts[0])
print("Example label:", labels[0])

Example text: Rahul Sharma is a Social Media Manager with 2 years of experience at Meta. Skills include SEO, Content Marketing, Google Analytics.
Example label: 11


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 480
Test size: 120


In [14]:
from config import USE_MODEL_URL

print("Loading encoder...")

encoder = hub.load(USE_MODEL_URL)

print("Encoder loaded")

Loading encoder...
Encoder loaded


In [15]:
print("Encoding training data...")

X_train_emb = encoder(X_train).numpy()
X_test_emb = encoder(X_test).numpy()

print("Train embeddings shape:", X_train_emb.shape)
print("Test embeddings shape:", X_test_emb.shape)

Encoding training data...
Train embeddings shape: (480, 512)
Test embeddings shape: (120, 512)


In [16]:
num_templates = len(set(labels))
print("Total templates:", num_templates)

Total templates: 50


In [17]:
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(512,)),
    tf.keras.layers.Dense(256, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(num_templates, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 50)             │         6,450 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 170,674 (666.70 KB)

 Trainable params: 170,674 (666.70 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
history = model.fit(
    X_train_emb,
    y_train,
    validation_data=(X_test_emb, y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0208 - loss: 3.9153 - val_accuracy: 0.0083 - val_loss: 3.9172
Epoch 2/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0417 - loss: 3.8894 - val_accuracy: 0.0083 - val_loss: 3.9205
Epoch 3/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.0500 - loss: 3.8650 - val_accuracy: 0.0083 - val_loss: 3.9348
Epoch 4/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0437 - loss: 3.8357 - val_accuracy: 0.0250 - val_loss: 3.9619
Epoch 5/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0417 - loss: 3.7985 - val_accuracy: 0.0167 - val_loss: 3.9815
Epoch 6/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.0479 - loss: 3.7606 - val_accuracy: 0.0083 - val_loss: 4.0214
Epoch 7/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0729 - loss: 3.7323 - val_accuracy: 0.0083 - val_loss: 4.0318
Epoch 8/10
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.0833 - loss: 3.6963 - val_accuracy: 0.0083 - val_loss

In [19]:
model_path = Path("../model/summary_model.keras")

model_path.parent.mkdir(parents=True, exist_ok=True)

model.save(model_path)

print("Model saved to:", model_path)

Model saved to: ..\model\summary_model.keras


In [20]:
import tensorflow as tf
from pathlib import Path

In [21]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

tflite_path = Path("../model/tflite")
tflite_path.mkdir(parents=True, exist_ok=True)

with open(tflite_path / "rsg_model.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved")

INFO:tensorflow:Assets written to: C:\Users\Iqrak\AppData\Local\Temp\tmp_gvn_jzk\assets


INFO:tensorflow:Assets written to: C:\Users\Iqrak\AppData\Local\Temp\tmp_gvn_jzk\assets


Saved artifact at 'C:\Users\Iqrak\AppData\Local\Temp\tmp_gvn_jzk'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 512), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 50), dtype=tf.float32, name=None)
Captures:
  1721854674544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1721854884176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1721741169008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1721741176048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1719753871200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1719754073792: TensorSpec(shape=(), dtype=tf.resource, name=None)
TFLite model saved
